# dmg777k: preprocessing

Prepares dmg777k for the experiments of Milestone 3. It follows the authors' pipeline (`original/data_preparation/dmg_data_prep.py` and `original/geo_rdf2vec.py`), keeps everything the experiments do not test the same as theirs, and fixes the problems the EDA found. It makes no modelling decisions; those are listed at the end.

**To run it:** prepare the data as in `data/README.md`, then run this notebook from top to bottom (about 2 minutes, about 6 GB of memory). The same steps run without the notebook with `uv run python -m preprocessing.pipeline dmg777k`.

In [1]:
import pandas as pd

from preprocessing import pipeline
from preprocessing.geometry import geodesic_km

pd.set_option("display.max_colwidth", 70, "display.width", 160)


def short(iri: str) -> str:
    """Last part of an IRI, for display only; the saved tables keep full IRIs."""
    return iri.rstrip("/").rsplit("/", 1)[-1].rsplit("#", 1)[-1]


r = pipeline.run("dmg777k")
print(f"{len(r.dataset.triples):,} triples loaded, {len(r.triples):,} kept")

777,124 triples loaded, 776,920 kept


## 1. Preparing the data

### 1.1 The authors' pipeline, with fixes

The steps are the authors':

1. **Load the triples** and replace integer IDs with node text and relation IRIs.
2. **Convert the 371 municipality outlines stored in RD New metres to WGS84**, as `dmg_data_prep.py` does.
3. **Build the graph** from the triples.
4. **Find the geometries** and their centres.

We change four things:

- **Missing values are removed explicitly.** The authors' `dropna()` drops triples whose object is the empty text, because pandas reads it as missing. We remove the same triples, but on purpose and with a list of placeholder texts (`preprocessing/graph.py`).
- **A node is identified by its text and its kind**, the kind being "entity" or the literal's datatype or language tag. The authors identify nodes by text alone.
- **A geometry is any literal typed as a WKT literal**, parsed in any letter case. The authors use a case-sensitive regular expression.
- **Coordinates are always (longitude, latitude).** The authors pass them to geopy in the order geopy reads as (latitude, longitude).

**Missing values.** Triples whose object is a placeholder text, grouped by that text:

In [2]:
removed = r.placeholders_removed.assign(predicate=r.placeholders_removed["predicate"].map(short))
print(f"Triples removed: {removed['triples'].sum():,}")
removed.groupby("object")["triples"].sum().sort_values(ascending=False).to_frame()

Triples removed: 204


,triples
object,
,204


**Interpretation:** the only missing value in dmg777k is the empty text, in 204 photo credits and dates. These are exactly the triples the authors' `dropna()` removes, so the graph keeps 776,920 triples, as in the paper.

**Node identity.** The graph with nodes identified by text (the authors) and by text and kind (ours):

In [3]:
r.identity

,By text (authors),By text and kind (ours)
Nodes,332194,341269
Self-loops,8396,0
Nodes where a literal is merged into an entity,8396,0
Nodes where literals of different types are merged,632,0


**Interpretation:** identifying nodes by text alone merges 9,075 nodes:

- **Every one of the 8,396 labelled monuments is merged with a literal.** Its `seeAlso` literal holds the same text as the monument's IRI, so the monument becomes linked to itself. Walks then step in place, and the EDA showed that such a step shifts the relation names in the walk text.
- **632 texts merge literals of different types.** For example, the monument number "10057" becomes the same node as a city's population of 10,057, which links a monument to a city that has nothing to do with it.

**With nodes identified by text and kind, none of this happens.** The graph has 341,269 nodes: KGBench's own 341,270, minus the empty text. KGBench already merged identical values of the same type, so identifying nodes by text adds only wrong merges.

**Geometries.** WKT literals, and how many hold a usable geometry:

In [4]:
print(r.geometry_counts.to_dict())
print("Geometry types:", r.geometries["geometry_type"].value_counts().to_dict())
print(f"Entities with their own geometry: {len(r.locations):,}")

{'WKT literals': 21208, 'Not valid WKT': 0, 'Empty': 0, 'Usable': 21208}
Geometry types: {'Polygon': 11215, 'Point': 9893, 'MultiPolygon': 100}
Entities with their own geometry: 31,169


**Interpretation:** all 21,208 WKT literals, including the 371 converted outlines, are valid geometries: 11,215 polygons, 9,893 points and 100 multipolygons. 31,169 entities have their own geometry: the monuments, through their geometry node, and the geometry nodes themselves. Every other node only gets a location through flooding.

### 1.2 Checks: the problems from the EDA are gone

Each row is a problem the EDA found. The first column shows the authors' pipeline, the second ours. The distance row runs our coordinate handling and the authors' order on two points with a known distance (geopy gives 34.6 km), written in both letter cases the data uses.

In [5]:
r.checks

,authors' pipeline,ours,expected
check,,,
WKT literals recognised as geometries,12851.0,21208.0,"all usable of 21,208"
Geometry centres outside the country,NaN,0.0,0
"Amsterdam–Utrecht distance, km (true 34.6)",39.4,34.6,34.6
Self-loops,8396.0,0.0,0
Nodes merging an entity and a literal,8396.0,0.0,0
Nodes merging literals of different types,632.0,0.0,0


**Interpretation:**

- **The authors' expression misses 8,357 geometries**, all written `Point(...)`. That is 39% of them. We recognise all of them.
- **All geometry centres lie in the Netherlands**, so the coordinate order is right.
- **Our distance is correct (34.6 km).** The authors' order gives 39.4 km, because a degree east is counted as a degree north.
- **There are no self-loops and no wrong merges left.**

## 2. Describing the relations

One row per relation: number of edges, distinct objects, whether it points to entities or literals (and which kind), and its spatial spread. The spread uses the entities that have their own geometry: for each object shared by at least two of them, it is the median distance of each to the group's mean position. A small value means the relation links things close together. Relations with fewer than 30 such entities get no value. This is the same measure as in the EDA. The table holds facts only; which relations pass on locations is decided in the experiments.

In [6]:
loc = r.locations
baseline_km = pd.Series(geodesic_km(loc["lon"], loc["lat"], loc["lon"].mean(), loc["lat"].mean())).median()
print(f"Baseline: median distance of a located entity to the mean position of all of them: {baseline_km:.1f} km")
relations = r.relations.assign(relation=r.relations["relation"].map(short), literal_kind=r.relations["literal_kind"].map(lambda k: k if pd.isna(k) else short(k)))
print(f"{len(relations)} relations, {relations['median_km'].notna().sum()} with a spatial spread")
relations.dropna(subset=["median_km"]).sort_values("median_km")

Baseline: median distance of a located entity to the mean position of all of them: 75.2 km
60 relations, 29 with a spatial spread


,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
12,asWKT,20866,20837,literal,wktLiteral,54.0,0.00
11,hasGeometry,21039,20871,entity,NaN,302.0,0.00
19,street-address,8396,8368,literal,plain,46.0,0.00
18,postal-code,8396,5846,literal,plain,3904.0,0.03
45,internComplexNummers,630,490,literal,plain,224.0,0.03
44,complexnummer,630,490,literal,plain,224.0,0.03
1,description,57042,33659,literal,@nl-nl,351.0,0.07
51,neighbourhood,258,93,literal,plain,205.0,0.27
21,city,8396,1536,entity,NaN,7793.0,0.37
53,location,156,99,literal,plain,76.0,0.67


**Interpretation:** the spread separates the relations clearly, against a baseline of 75 km.

- **Spatial:** street address, postal code, complex number and neighbourhood link places within metres, and `city` within 0.4 km. `sfWithin` is at 1.9 km.
- **Regional:** `parentADM2` (3.3 km) and `parentADM1` (22 km).
- **Not spatial:** `type`, house numbers, building years and the country-level GeoNames codes are near the baseline (66–88 km). Shared house numbers and years group monuments from all over the country.
- **Literal relations are spatial too.** Postal codes and street names behave like places, so a flooding rule that ignored literals would miss them.

The full table, including relations without enough located subjects, is saved as `relations.csv`:

In [7]:
relations

,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
0,type,130126,7,entity,NaN,23515.0,75.90
1,description,57042,33659,literal,@nl-nl,351.0,0.07
2,depicts,47872,8396,entity,NaN,NaN,NaN
3,depiction,47872,47872,entity,NaN,NaN,NaN
4,isPartOf,47384,38,literal,@nl-nl,NaN,NaN
5,locator,46123,46091,literal,anyURI,NaN,NaN
6,thumbnail,46108,46061,literal,base64Image,NaN,NaN
7,creator,45009,626,literal,plain,NaN,NaN
8,created,44215,1081,literal,plain,NaN,NaN
9,fotograaf,43863,463,literal,plain,NaN,NaN


### Hubs

The nodes with the most edges, and the relation that brings most of them. In flooding, a hub collects the geometries of everything linked to it and passes the mix on. That is why the EDA found 18.5% of nodes at one national centroid.

In [8]:
hubs = r.hubs.assign(node=r.hubs["node"].str[-60:], kind=r.hubs["kind"].map(short), main_relation=r.hubs["main_relation"].map(short))
hubs.head(15)

,node,kind,degree,in_degree,out_degree,main_relation,main_relation_share
0,http://www.opengis.net/ont/geosparql#Geometry,entity,65576,65576,0,type,1.000
1,https://data.labs.pdok.nl/rce/def/Afbeelding,entity,60996,60996,0,type,1.000
2,"""zwart wit negatief""@nl-nl",@nl-nl,23232,23232,0,techniek,1.000
3,"""Fotocollectie""@nl-nl",@nl-nl,18146,18146,0,isPartOf,1.000
4,"""Collectie gebouwd""@nl-nl",@nl-nl,16868,16868,0,isPartOf,1.000
5,"""Dukker, G.J.""",plain,13544,13544,0,creator,0.500
6,"""Wal, A.J. van der""",plain,9396,9396,0,creator,0.500
7,"""true""^^<http://www.w3.org/2001/XMLSchema#boolean>",boolean,8230,8230,0,isFree,1.000
8,"""Agrarisch Erfgoed - Fotocollectie SHBO""@nl-nl",@nl-nl,8090,8090,0,isPartOf,1.000
9,"""Digitale opname""@nl-nl",@nl-nl,6023,6023,0,techniek,1.000


**Interpretation:** the largest hubs are the classes `Geometry` and `Afbeelding` ("image"), each reached only through `type`, and shared literals describing photographs: the technique ("zwart wit negatief", black-and-white negative), the collection and the photographer. None of them is a place. The list is saved as `hubs.csv`, for the experiments to decide what to do with them.

## 3. Labels and splits

The official KGBench splits. Every check should be 0.

In [9]:
display(r.splits)
r.label_checks

split,train,valid,test,total
class,,,,
0,205,38,76,319
1,2578,478,956,4012
2,387,72,144,603
3,1496,278,555,2329
4,728,135,270,1133
total,5394,1001,2001,8396


,count
Nodes labelled more than once,0
Labelled nodes missing from the graph,0
"Triples using the target relation (monumentCode, http://purl.org/dc/terms/subject)",0


**Interpretation:** 8,396 monuments in 5 classes, with the same class shares in each split. No monument appears twice, all are in the graph, and the target relation is absent, so no label leaks into the graph.

## 4. Saving

All outputs go to `data/kgbench_dmg777k/processed/` (git-ignored):

| File | Contents |
|---|---|
| `triples.parquet` | subject, predicate, object, object_kind. Subjects and objects are node keys: the IRI for an entity; for a literal, its text in quotes with its datatype or language tag, as in N-Triples |
| `geometries.parquet` | node key, WKT, geometry type and centre (lon, lat) of every WKT literal |
| `labels.csv` | node, class, split |
| `relations.csv` | the relation table of Section 2 |
| `hubs.csv` | the 50 nodes with the most edges |

In [10]:
pipeline.save(r)
for path in sorted(r.dataset.processed_dir.iterdir()):
    print(f"{path.name:20} {path.stat().st_size / 1e6:8.1f} MB")

geometries.parquet      105.7 MB
hubs.csv                  0.0 MB
labels.csv                0.5 MB
relations.csv             0.0 MB
triples.parquet         580.9 MB


## What is left to the experiments

Preprocessing makes no modelling decisions. These belong to the experiments (see `experiments/README.md`):

- **Walk direction and walk start.** Triples are stored in their original direction. Whether walks may also follow edges backwards, and which nodes they start from, are walker settings.
- **Which relations pass on a location during flooding, and what to do with hubs.** The relation table and the hub list above are the facts these choices are based on. The choices themselves live in experiment configurations.
- **Flooding and edge weights.** `geometries.parquet` holds the starting geometries. Flooding them through the graph is the experiment.
